# Adaptive RAG on Colab

This notebook stands my thesis system up on a fresh Colab GPU runtime and runs
it end to end: clone the repo, install the CUDA stack, prepare a few datasets,
build an index, smoke-test the pipeline offline, then do a small real
evaluation with the 7B generator. The system is a zero-shot LLM router over
three retrieval tiers with a corrective gate that can promote a query one tier
up when the evidence looks wrong.

Runtime: GPU, T4 is enough. Set it before running anything
(Runtime, Change runtime type, Hardware accelerator: GPU).


## 1. Environment and GPU check

If the cell below shows no NVIDIA GPU, switch the runtime first and rerun
from the top. T4 is the default and enough for everything here; L4 or A100
only help for the long evaluation sweeps.


In [ ]:
import os, shutil, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
# TPU runtimes expose one of these; see the TPU section at the end of the notebook.
ON_TPU = bool(os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_ACCELERATOR_TYPE")
              or os.environ.get("TPU_NAME"))

print(f"python : {sys.version.split()[0]}")
print(f"colab  : {IN_COLAB} | kaggle: {IN_KAGGLE} | tpu runtime: {ON_TPU}")

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found -- no NVIDIA GPU visible on this runtime.")

try:
    import torch
    print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  cuda:{i} {torch.cuda.get_device_name(i)}")
        free, total = torch.cuda.mem_get_info()
        print(f"  vram: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
        cap = torch.cuda.get_device_capability(0)
        if cap < (8, 0):
            print(f"  note: compute capability {cap} < (8,0): no native bfloat16 "
                  "(T4/P100). Works, but bf16 ops are emulated and slower.")
except ImportError:
    print("torch not importable yet -- it is installed in the setup cell below.")

if ON_TPU:
    print("\n*** TPU runtime detected. This pipeline targets CUDA -- please switch "
          "to a GPU runtime (see the TPU section at the bottom for why). ***")
elif not shutil.which("nvidia-smi"):
    print("\nWARNING: no GPU. The 7B generator would fall back to CPU and be "
          "unusably slow. Switch the runtime to GPU before continuing.")


## 2. Get the repo

The repo lives at github.com/abym-droid/zero-shot-adaptive-corrective-rag
(private). Two ways in:

* clone: keep `REPO_URL` as set below; for a private repo use a token URL
  (`https://<user>:<token>@github.com/...`) or add the token via getpass.
* zip upload: set `REPO_URL` to "" and upload
  `zero-shot-adaptive-corrective-rag.zip` through the file browser on the
  left; the cell finds the zip, extracts it and locates the folder with
  `pyproject.toml`.


In [ ]:
import subprocess, zipfile
from pathlib import Path

# ---- parameters -------------------------------------------------------------
REPO_URL = "https://github.com/abym-droid/zero-shot-adaptive-corrective-rag.git"  # "" -> zip fallback
WORK_DIR = Path("/content")
REPO_DIR = WORK_DIR / "zero-shot-adaptive-corrective-rag"
# ------------------------------------------------------------------------------

def _find_repo_root(extract_dir: Path) -> Path:
    """Locate the folder holding pyproject.toml (zips often nest one level)."""
    if (extract_dir / "pyproject.toml").exists():
        return extract_dir
    hits = sorted(extract_dir.glob("**/pyproject.toml"))
    if not hits:
        raise FileNotFoundError(f"no pyproject.toml under {extract_dir}")
    return hits[0].parent

if (REPO_DIR / "pyproject.toml").exists():
    print(f"repo already present: {REPO_DIR}")
elif REPO_URL:
    # -- A. git clone ----------------------------------------------------------
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    # -- B. zip fallback ------------------------------------------------------
    zips = sorted(Path(WORK_DIR).glob("*.zip"))
    if not zips and IN_COLAB:
        from google.colab import files  # upload widget
        print("No REPO_URL and no zip found -- upload zero-shot-adaptive-corrective-rag.zip now:")
        uploaded = files.upload()
        zips = [Path(n).resolve() for n in uploaded if n.endswith(".zip")]
    if not zips:
        raise FileNotFoundError(
            "Neither REPO_URL set nor a repo zip found. Set REPO_URL above, or "
            "provide the zip (/content) and re-run this cell.")
    zip_path = zips[0]
    print(f"extracting {zip_path} ...")
    extract_to = WORK_DIR / "_repo_zip"
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)
    root = _find_repo_root(extract_to)
    REPO_DIR.exists() or root.rename(REPO_DIR)
    print(f"repo extracted to {REPO_DIR}")

import os
os.chdir(REPO_DIR)  # all repo-relative paths (data/, indices/, runs/) resolve from here
print("cwd:", os.getcwd())


## 3. Install the CUDA dependency stack

Same as `environment/requirements-core.txt` minus the Apple-only packages:
no mlx here. Generation goes through the HF backend, which picks up
bitsandbytes 4-bit on CUDA, so the model runs at the same reduced precision
as the MLX path on my laptop. Torch ships preinstalled with CUDA on Colab
and is left alone.


In [ ]:
%pip install -q "transformers>=4.56" "accelerate>=0.33" "sentence-transformers>=3.0" \
    "datasets>=2.20" "huggingface_hub>=0.24" "faiss-cpu>=1.8" "rank-bm25>=0.2.2" \
    "langgraph>=0.2" "langchain-core>=0.2" "pydantic>=2.7" "pydantic-settings>=2.3" \
    "typer>=0.12" rich tqdm jsonlines orjson pyyaml evaluate rouge-score \
    "bitsandbytes>=0.43"
%pip install -q -e .


In [ ]:
# Import probe: the editable install must resolve in THIS kernel without a restart.
import sys
from pathlib import Path

src = str(Path("src").resolve())
if src not in sys.path:           # belt-and-braces for editable installs
    sys.path.insert(0, src)

from adarag.config import settings
from adarag.device import get_device

print("adarag import OK")
print("resolved device      :", get_device(settings.device))   # expect: cuda
print("max_escalations bound:", settings.max_escalations, "(one tier promotion max)")

# Environment / settings / model-cache report from the CLI entry point:
!adarag info


### 3a. Optional: Hugging Face login

Only needed for the gated telecom sources (teleqna, tspec-llm) after
accepting their terms on the Hub. Everything else in this notebook runs
without a token. Put the token in Colab Secrets as HF_TOKEN (key icon in
the left sidebar), then uncomment below.


In [ ]:
# gated telecom sets need an HF token + accepted terms; skipped in this run
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# !hf download rasoul-nikbakht/TSpec-LLM --repo-type dataset --include "3GPP-clean/Rel-16/*.md" --include "3GPP-clean/Rel-17/*.md" --local-dir data/raw/tspec_llm


## 4. Get the datasets

Raw files are fetched straight from their sources into `data/raw/`, then
`scripts/prepare_datasets.py` normalises them and cuts the seeded dev500 and
toy slices. Same script and seed as on my laptop, so the slices match
exactly.

Here I take squad (single-hop), hotpotqa (multi-hop, gives the router a
reason to pick the iterative tier) and scifact, which also provides the
retrieval corpus indexed in the next step.


In [ ]:
%%bash
mkdir -p data/raw/squad data/raw/hotpotqa data/raw/scifact
H=https://huggingface.co/datasets
wget -nc -q -O data/raw/squad/validation.parquet    $H/rajpurkar/squad/resolve/main/plain_text/validation-00000-of-00001.parquet
wget -nc -q -O data/raw/hotpotqa/validation.parquet $H/hotpotqa/hotpot_qa/resolve/main/distractor/validation-00000-of-00001.parquet
wget -nc -q -O data/raw/scifact/claims.parquet      $H/allenai/scifact/resolve/refs%2Fconvert%2Fparquet/claims/validation/0000.parquet
wget -nc -q -O data/raw/scifact/corpus.parquet      $H/allenai/scifact/resolve/refs%2Fconvert%2Fparquet/corpus/train/0000.parquet
python scripts/prepare_datasets.py squad hotpotqa scifact
echo; echo '--- processed files ---'; ls -la data/processed/


## 5. Build retrieval indices

`adarag index` chunks a corpus jsonl and writes a BM25 index and a FAISS
dense index (MiniLM embeddings, encoded on the GPU here). Two indices: a
tiny inline telecom corpus for the offline smoke test, and the real SciFact
corpus for the evaluation run.


In [ ]:
import json
from pathlib import Path

TOY_DOCS = [
    {"doc_id": "toy-001", "title": "3GPP and 5G NR",
     "text": "The 3rd Generation Partnership Project (3GPP) publishes the 5G New Radio "
             "(NR) specifications. 5G NR was first standardised in 3GPP Release 15, with "
             "further enhancements in Release 16 and Release 17."},
    {"doc_id": "toy-002", "title": "OSS and BSS",
     "text": "In telecom operations, OSS (Operations Support Systems) manage network "
             "inventory, provisioning and fault management, while BSS (Business Support "
             "Systems) handle billing, CRM and order management."},
    {"doc_id": "toy-003", "title": "Retrieval-Augmented Generation",
     "text": "Retrieval-Augmented Generation (RAG) grounds a language model's answer in "
             "passages fetched from an external corpus, reducing hallucination on "
             "knowledge-intensive questions."},
    {"doc_id": "toy-004", "title": "BM25",
     "text": "BM25 is a sparse lexical ranking function based on term frequency, inverse "
             "document frequency and document length normalisation. It remains a strong "
             "retrieval baseline."},
    {"doc_id": "toy-005", "title": "Dense retrieval",
     "text": "Dense retrieval encodes queries and passages into a shared embedding space "
             "and ranks passages by vector similarity, typically with a FAISS index."},
    {"doc_id": "toy-006", "title": "Adaptive-RAG",
     "text": "Adaptive-RAG routes each query to a retrieval strategy matched to its "
             "complexity: no retrieval, single-step retrieval, or iterative multi-step "
             "retrieval."},
    {"doc_id": "toy-007", "title": "CRAG",
     "text": "Corrective RAG (CRAG) evaluates the quality of retrieved evidence and "
             "triggers a corrective action when the evidence is judged incorrect."},
    {"doc_id": "toy-008", "title": "Network slicing",
     "text": "5G network slicing partitions one physical network into multiple virtual "
             "end-to-end networks, each tailored to a service class such as eMBB, URLLC "
             "or mMTC."},
    {"doc_id": "toy-009", "title": "ColBERTv2",
     "text": "ColBERTv2 is a late-interaction retriever: it keeps token-level embeddings "
             "and computes MaxSim interactions between query and document tokens at "
             "search time."},
    {"doc_id": "toy-010", "title": "IRCoT",
     "text": "IRCoT interleaves retrieval with chain-of-thought reasoning: each reasoning "
             "step issues a new retrieval query, supporting multi-hop questions."},
]

Path("data/toy").mkdir(parents=True, exist_ok=True)
with open("data/toy/corpus.jsonl", "w", encoding="utf-8") as fh:
    for d in TOY_DOCS:
        fh.write(json.dumps(d, ensure_ascii=False) + "\n")
print(f"wrote {len(TOY_DOCS)} toy docs -> data/toy/corpus.jsonl")

!adarag index data/toy/corpus.jsonl --out-dir data/toy/indices


In [ ]:
# SciFact corpus index (~5k abstracts; MiniLM encoding runs on the GPU, ~1 min).
!adarag index data/processed/scifact.corpus.jsonl --out-dir indices/scifact

print("\n--- index layout ---")
!find indices/scifact -maxdepth 2 -type f | sort


## 6. Pipeline smoke test, offline

Before spending GPU time I check the whole loop, route then execute then
gate then escalate, with the deterministic fake backend. No model downloads,
no GPU. This is the same offline path the test suite uses.


In [ ]:
!adarag ask "Which organisation publishes the 5G NR specifications?" --fake

# Harness smoke: 10 SQuAD dev500 examples through the eval loop on canned responses.
!adarag eval --dataset data/processed/squad.dev500.jsonl --backend fake --limit 10 \
    --out-dir runs/fake_smoke


## 7. Real evaluation run

Now the real thing: `adarag eval` over the SciFact dev500 slice with the
BM25 index and the HF backend (Qwen2.5-7B-Instruct in 4-bit, about 5.5 GB
VRAM, fits a T4). Each example records the routing decision, gate verdict,
escalations, EM/F1, latency and token counts; the run directory gets
predictions.jsonl and summary.json.

Notes:
* the first run downloads about 15 GB of model weights, cached afterwards
* `--limit 20` keeps this to 10-20 minutes on a T4; a thesis-grade run drops
  the limit and needs a few GPU hours
* ablation flags: `--no-escalation`, `--prompt-variant v2`,
  `--retriever dense`


In [ ]:
RUN_DIR = "runs/colab_scifact_hf"

!adarag eval --dataset data/processed/scifact.dev500.jsonl --index indices/scifact \
    --backend hf --retriever bm25 --prompt-variant v1 --limit 20 \
    --out-dir $RUN_DIR


In [ ]:
import json
from pathlib import Path

run_dir = Path(RUN_DIR)
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
print("=== summary.json ===")
print(json.dumps(summary, indent=2))

# Per-example view: routing tiers, escalations, gate verdicts, scores.
import pandas as pd

rows = [json.loads(l) for l in (run_dir / "predictions.jsonl").read_text(
    encoding="utf-8").splitlines() if l.strip()]
df = pd.DataFrame(rows)[
    ["qid", "tier_initial", "tier_final", "escalated", "verdict",
     "em", "f1", "latency_s", "prompt_tokens", "completion_tokens"]
]
display(df.head(20))
print("\ntier distribution (initial):", df["tier_initial"].value_counts().to_dict())
print("escalation rate            :", round(float(df["escalated"].mean()), 3))


## 8. Why not TPU

Colab also offers TPU runtimes; this pipeline cannot use them. Device
selection resolves mps, cuda or cpu, so on a TPU runtime the generator would
silently land on CPU and crawl. Reaching the TPU would need torch_xla, which
this stack does not integrate, and bitsandbytes 4-bit is CUDA-only, so the
reduced-precision setting has no TPU equivalent. The thesis claim is about
commodity local hardware anyway. Use a GPU runtime.


## 9. Next steps

* full runs: drop `--limit` and loop over the dev500 files, one run
  directory per dataset
* ablation grid: escalation on and off, prompt variant v1 and v2, bm25 and
  dense
* telecom domain: accept the gated terms, run cell 3a, download, then index
  the tspec corpus
* routing accuracy against silver labels is already in summary.json
* copy results off the VM before it recycles: zip the runs directory and
  download it, or mount Drive and copy it there
